# **Single-Cell RNA-Seq Analysis Project**

In this project, you will work with the `norman` dataset from the `perturbation_data_analysis` exercise:

In [2]:
import os
import sys
from pathlib import Path
import scanpy as sc

project_root = Path().absolute().parent
# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

import pertdata as pt  # noqa: E402

# Use the existing norman dataset location
norman = pt.PertDataset(name="norman", cache_dir_path="../data", silent=False)

print(norman)

Downloading: https://dataverse.harvard.edu/api/access/datafile/6154020 -> d:\IML\Genomic-Data-Science\data\norman\data.zip
Total size: 168,758,985 bytes


100%|██████████| 169M/169M [02:18<00:00, 1.22MiB/s] 


Download completed: d:\IML\Genomic-Data-Science\data\norman\data.zip
Loading: d:\IML\Genomic-Data-Science\data\norman\norman\perturb_processed.h5ad
PertDataset object
    name: norman
    cache_dir_path: d:\IML\Genomic-Data-Science\data
    path: d:\IML\Genomic-Data-Science\data\norman
    adata: AnnData object with n_obs ✕ n_vars = 91205 ✕ 5045


In [4]:

adata = sc.read_h5ad('../data/norman/norman/perturb_processed.h5ad')
print(f"Dataset loaded: {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"Available metadata: {list(adata.obs.columns)}")

Dataset loaded: 91205 cells × 5045 genes
Available metadata: ['condition', 'cell_type', 'dose_val', 'control', 'condition_name']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from src.models.svm_model import SVMModel
import numpy as np

# Keep data SPARSE - don't convert to dense array!
X = adata.X  # Keep as sparse matrix
print(f"Gene expression matrix: {X.shape} (sparse: {type(X).__name__})")

y_labels = adata.obs['condition'].values
print(f"Labels shape: {y_labels.shape}")
print(f"Unique perturbations: {len(set(y_labels))}")
print(f"Sample labels: {y_labels[:5]}")

# Encode string labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)
print(f"\nEncoded labels range: {y.min()} to {y.max()}")

# Split into train/test (80/20) - NO SUBSAMPLING, using full dataset
print("\n" + "="*50)
print("Using FULL dataset (sparse format)")
print("="*50)
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_seed, stratify=y
)

print(f"\nTraining data: X_train.shape={X_train.shape}")
print(f"Test data: X_test.shape={X_test.shape}")
print(f"Memory usage: ~{X_train.data.nbytes / 1e6:.1f} MB (sparse)")

# Create and train SVM with sparse support
print("\n" + "="*50)
print("Training LinearSVM (memory efficient)...")
print("="*50)
model = SVMModel(
    kernel='linear',      # LinearSVC is much faster & memory efficient
    C=1.0, 
    pca_components=100,   # Dimensionality reduction
    use_sparse=True,      # Don't center sparse matrices
    random_state=random_seed
)
model.fit(X_train, y_train)

# Evaluate on test set
print("\nEvaluating on test set...")
results = model.evaluate(X_test, y_test)
print("\nTest Results:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

Gene expression matrix: (91205, 5045) (sparse: csr_matrix)
Labels shape: (91205,)
Unique perturbations: 284
Sample labels: ['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl']
Categories (284, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

Encoded labels range: 0 to 283

Using FULL dataset (sparse format)

Training data: X_train.shape=(72964, 5045)
Test data: X_test.shape=(18241, 5045)
Memory usage: ~119.4 MB (sparse)

Training LinearSVM (memory efficient)...


ValueError: Cannot center sparse matrices: pass `with_mean=False` instead. See docstring for motivation and alternatives.

Choose one of the following tasks:

**Exploratory Data Analysis and Visualization**

- Objective: Explore the dataset to identify patterns and clusters.
- Tasks:
  - Perform dimensionality reduction using PCA, t-SNE, or UMAP.
  - Visualize gene expression profiles across different conditions or perturbations.
  - Create heatmaps of the top differentially expressed genes.
- Learning Outcomes:
  - Learn to visualize high-dimensional data.
  - Interpret clusters and patterns in the context of biological conditions.

**Machine Learning Classification**

- Objective: Build models to classify samples based on gene expression profiles.
- Tasks:
  - Split the dataset into training and testing sets.
  - Implement classification algorithms.
  - Evaluate model performance using metrics like accuracy, precision, recall, and ROC curves.
- Learning Outcomes:
  - Understand supervised learning techniques.
  - Learn model evaluation and validation strategies.

**Advanced Deep Learning Applications**

- Objective: Apply deep learning techniques to model complex patterns in the data.
- Tasks:
  - Implement autoencoders or variational autoencoders for dimensionality reduction.
  - Explore the use of GANs to generate synthetic gene expression data.
  - Analyze how deep learning models capture nonlinear relationships.
- Learning Outcomes:
  - Gain experience with deep learning frameworks.
  - Understand the applications of deep learning in genomics.